# This Evaluates the Retrieval and Response Generation of the RAG model

In [3]:
import time
import pandas as pd

from utils import (
    get_embedding,
    cosine_similarity,
    df as docs_df,
    client,
    CHAT_MODEL,
    TEMPERATURE,
    MAX_TOKENS,
    TOP_K,
    SIMILARITY_THRESHOLD,
)

# 1) Load your test set
test_df = pd.read_excel("testset.xlsx")  # columns: id, file, chapter, question, expected_answer

# Prepare logs
metrics = []
responses = []

# System prompt (same as in your RAG system)
SYSTEM_PROMPT = (
    "Du bist ein hilfreicher und professioneller KI-Assistent. "
    "Du beantwortest ausschliesslich Fragen zum Lehrbuch \"Wissenschaftliches Arbeiten und Kommunizieren\". "
    "Deine Antworten sollen klar, korrekt, präzise und fachlich fundiert sein. "
    "Falls die gestellte Frage nicht auf Basis des Kontexts beantwortet werden kann, sage: "
    "\"Ich bin mir nicht sicher. Kannst du deine Frage umformulieren?\" "
    "Erfinde keine Informationen und nutze ausschliesslich den bereitgestellten Kontext."
)

for _, row in test_df.iterrows():
    qid             = row["id"]
    expected_file   = row["file"]
    expected_chap   = row["chapter"]
    question        = row["question"]

    # --- Embedding ---
    t0 = time.perf_counter()
    user_emb = get_embedding(question)
    t1 = time.perf_counter()
    embed_time = t1 - t0

    # --- Retrieval ---
    t2 = time.perf_counter()
    sims = docs_df["embedding"].apply(lambda x: cosine_similarity(x, user_emb))
    top_idxs = sims.nlargest(TOP_K).index
    retrieval_time = time.perf_counter() - t2

    # Check retrieval accuracy
    retrieved_files    = docs_df.loc[top_idxs, "filename"].tolist()
    retrieved_chapters = docs_df.loc[top_idxs, "section_title"].tolist()
    retrieval_correct = (
        expected_file in retrieved_files or expected_chap in retrieved_chapters
    )

    # Build context for response
    context = "\n\n".join(docs_df.loc[top_idxs, "content"].tolist())

    # --- Response Generation ---
    t3 = time.perf_counter()
    chat_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system",  "content": SYSTEM_PROMPT},
            {"role": "user",    "content": f"Kontext: {context}\n\nFrage: {question}\nAntwort:"}
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    response_text = chat_resp.choices[0].message.content.strip()
    response_time = time.perf_counter() - t3

    # Log metrics & response
    metrics.append({
        "id":                qid,
        "embed_time_s":      embed_time,
        "retrieval_time_s":  retrieval_time,
        "response_time_s":   response_time,
        "retrieval_correct": retrieval_correct,
    })
    responses.append({
        "id":       qid,
        "response": response_text,
    })

# 4) Save out
pd.DataFrame(metrics).to_csv("evaluation_metrics.csv", index=False)
pd.DataFrame(responses).to_csv("generated_responses.csv", index=False)

print("✅ Evaluation done. Metrics → evaluation_metrics.csv; Responses → generated_responses.csv")


✅ Evaluation done. Metrics → evaluation_metrics.csv; Responses → generated_responses.csv


In [2]:
import pandas as pd
import time
from utils import get_response, get_embedding

# Load test set
TESTSET_PATH = "testset.xlsx"
K = 3  # Top-K retrieved documents

df_test = pd.read_excel(TESTSET_PATH)

# Store results
results = []

for idx, row in df_test.iterrows():
    question_id = row["id"]
    question = row["question"]
    expected_file = row["file"]  # Expected document filename

    # Measure embedding time
    start_embedding = time.time()
    _ = get_embedding(question)
    end_embedding = time.time()
    embedding_time = end_embedding - start_embedding

    # Measure full response time
    start_response = time.time()
    response, references = get_response(question)
    end_response = time.time()
    response_time = end_response - start_response

    # Check Recall@K
    retrieved_files = [ref["filename"] for ref in references] if references else []
    retrieved_correct = expected_file in retrieved_files

    results.append({
        "id": question_id,
        "question": question,
        "expected_file": expected_file,
        "retrieved_files": retrieved_files,
        "recall@K": retrieved_correct,
        "embedding_time_sec": round(embedding_time, 3),
        "response_time_sec": round(response_time, 3),
    })

# Convert to DataFrame and save
results_df = pd.DataFrame(results)
results_df.to_csv("evaluation_results.csv", index=False)

# Show summary
recall_score = results_df["recall@K"].mean()
print(f"Recall@{K}: {recall_score:.2%}")
print("Average embedding time:", results_df["embedding_time_sec"].mean(), "s")
print("Average response time:", results_df["response_time_sec"].mean(), "s")

results_df.head()


2025-06-13 09:47:22.796 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-13 09:47:22.996 
  command:

    streamlit run c:\Users\dubra\Documents\gitRepos\chat-vlscript\venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-06-13 09:47:22.997 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


An error occurred during embedding: Connection error.
An error occurred during embedding: Connection error.
An error occurred during embedding: Connection error.
An error occurred during embedding: Connection error.
Recall@3: 88.46%
Average embedding time: 1.387128205128205 s
Average response time: 3.078653846153846 s


,id,question,expected_file,retrieved_files,recall@K,embedding_time_sec,response_time_sec
0,1,Um was geht es hier?,01-introduction.qmd,"[07-structure.qmd, 14-poster.qmd, 10-writing.qmd]",False,1.463,2.003
1,2,Wie sieht der Alltag eines Forschers aus?,01-introduction.qmd,"[01-introduction.qmd, 02-researchprocess.qmd, ...",True,0.720,1.262
2,3,Was ist der Unterschied zwischen Grundlagenfor...,01-introduction.qmd,"[01-introduction.qmd, 10-writing.qmd, 02-resea...",True,0.315,3.744
3,4,Was ist der Unterschied zwischen Studie und Ex...,02-researchprocess.qmd,"[02-researchprocess.qmd, 04-datacollection.qmd...",True,0.294,3.187
4,5,Was ist der Unterschied zwischen Reproduzierba...,02-researchprocess.qmd,"[02-researchprocess.qmd, 08-content.qmd, 08-co...",True,0.410,4.095


## Evaluate Embedding speed
1. Embed each question in the test set and measure the time taken.
2. Save the embeddings in the test set.

1. Embed each question in the test set
2. Retrieve the top 3 documents using the RAG retrieval function
3. Compare top-1 retrieved filename to the ground truth file column
4. Score as 1 if correct, else 0
5. Compute the mean score

In [1]:
import pandas as pd
import numpy as np
from numpy.linalg import norm
from tqdm import tqdm
from utils import get_embedding, df as embeddings_df

# Load test set
testset = pd.read_excel("testset.xlsx")

# Define safe division function
def safe_divide(a, b):
    return a / b if b != 0 else 0

# Function to compute cosine similarity
def compute_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / safe_divide(norm(vec1) * norm(vec2), 1)

# Store results
results = []

# Evaluate retrieval for each question
for idx, row in tqdm(testset.iterrows(), total=len(testset)):
    question = row["question"]
    expected_file = row["file"]

    # Embed the question
    user_embedding = get_embedding(question, model='text-embedding-3-small')
    user_embedding_np = np.array(user_embedding)

    # Compute cosine similarities
    similarities = embeddings_df['embedding'].apply(
        lambda x: compute_similarity(x, user_embedding_np)
    )

    # Get top 3 most similar documents
    top_indices = similarities.nlargest(3).index
    top_filenames = embeddings_df.loc[top_indices, 'filename'].tolist()

    # Evaluate top-N retrieval
    is_correct_top1 = int(expected_file == top_filenames[0])
    is_correct_top2 = int(expected_file in top_filenames[:2])
    is_correct_top3 = int(expected_file in top_filenames[:3])

    # Save the result
    results.append({
        "question": question,
        "expected_file": expected_file,
        "top1_file": top_filenames[0],
        "top2_file": top_filenames[1] if len(top_filenames) > 1 else "",
        "top3_file": top_filenames[2] if len(top_filenames) > 2 else "",
        "top3_files": top_filenames,
        "is_correct_top1": is_correct_top1,
        "is_correct_top2": is_correct_top2,
        "is_correct_top3": is_correct_top3
    })

# Create a results DataFrame
results_df = pd.DataFrame(results)

# Calculate mean scores
top1_accuracy = results_df["is_correct_top1"].mean()
top2_accuracy = results_df["is_correct_top2"].mean()
top3_accuracy = results_df["is_correct_top3"].mean()

# Print results
print(f"Top-1 Retrieval Accuracy: {top1_accuracy:.2f}")
print(f"Top-2 Retrieval Accuracy: {top2_accuracy:.2f}")
print(f"Top-3 Retrieval Accuracy: {top3_accuracy:.2f}")

# Save results to CSV
results_df.to_csv("retrieval_evaluation_results.csv", index=False)
print("Detailed results saved to retrieval_evaluation_results.csv")

# Display DataFrame inline
try:
    import ace_tools as tools
    tools.display_dataframe_to_user(name="Retrieval Evaluation Results", dataframe=results_df)
except ImportError:
    from IPython.display import display
    display(results_df.head())


100%|██████████| 78/78 [00:24<00:00,  3.13it/s]

Top-1 Retrieval Accuracy: 0.78
Top-2 Retrieval Accuracy: 0.86
Top-3 Retrieval Accuracy: 0.91
Detailed results saved to retrieval_evaluation_results.csv


,question,expected_file,top1_file,top2_file,top3_file,top3_files,is_correct_top1,is_correct_top2,is_correct_top3
0,Um was geht es hier?,01-introduction.qmd,07-structure.qmd,14-poster.qmd,10-writing.qmd,"[07-structure.qmd, 14-poster.qmd, 10-writing.qmd]",0,0,0
1,Wie sieht der Alltag eines Forschers aus?,01-introduction.qmd,01-introduction.qmd,02-researchprocess.qmd,07-structure.qmd,"[01-introduction.qmd, 02-researchprocess.qmd, ...",1,1,1
2,Was ist der Unterschied zwischen Grundlagenfor...,01-introduction.qmd,01-introduction.qmd,10-writing.qmd,02-researchprocess.qmd,"[01-introduction.qmd, 10-writing.qmd, 02-resea...",1,1,1
3,Was ist der Unterschied zwischen Studie und Ex...,02-researchprocess.qmd,02-researchprocess.qmd,04-datacollection.qmd,08-content.qmd,"[02-researchprocess.qmd, 04-datacollection.qmd...",1,1,1
4,Was ist der Unterschied zwischen Reproduzierba...,02-researchprocess.qmd,02-researchprocess.qmd,08-content.qmd,08-content.qmd,"[02-researchprocess.qmd, 08-content.qmd, 08-co...",1,1,1
